# Track B on Colab GPU — independent regeneration + agreement classifier

For every closed-book row, TigerLLM-9B (4-bit) answers `prompt_bn` on its own — never seeing
`response_bn` — then we measure agreement (LaBSE cosine, two-way NLI, token/digit overlap)
and train a small classifier on the 169 labeled closed-book rows.

1. mounts Drive, clones/pulls the repo, restores caches
2. generates independent answers for train (169) + test (1,155) closed-book rows — **the slow part, ~1–2h total on a T4; resumable**, rerun the cell if Colab disconnects
3. computes agreement features, runs combined repeated CV (honest check BEFORE submitting)
4. builds + validates `submissions/nli_ctx_trackb_cb.csv`
5. copies all caches + the submission back to Drive

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Bengali-LLM-Hallucination-Detection'
import os
assert os.path.exists(f'{DRIVE_DIR}/data/test set.csv'), 'competition data not found on Drive'

In [ ]:
%cd /content
!git clone https://github.com/FaiyazAwsaf/Bengali-LLM-Hallucination-Detection.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo

!mkdir -p data submissions "{DRIVE_DIR}/cache"
!cp "{DRIVE_DIR}/data/dataset samples.json" "{DRIVE_DIR}/data/test set.csv" "{DRIVE_DIR}/data/sample submission.csv" data/
# cache lives DIRECTLY on Drive via symlink: every per-batch flush during generation is
# instantly persisted, so a runtime disconnect can never lose more than one batch
!rm -rf data/cache && ln -s "{DRIVE_DIR}/cache" data/cache
!ls data/cache/

!pip install -q -U transformers sentencepiece bitsandbytes accelerate sentence-transformers

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable a GPU runtime')

In [ ]:
# generate independent answers for the 169 labeled closed-book rows (~10-20 min)
# (cache is on Drive via the symlink — checkpointed automatically every batch)
%cd /content/repo
!cd src && python track_b_consistency.py gen train

In [ ]:
# eyeball a few generations before burning an hour on the test set:
# answers should be in Bengali and actually address the question
%cd /content/repo
import pandas as pd, json
gen = pd.read_csv('data/cache/tigerllm_train_answers.csv', index_col=0)
rows = json.load(open('data/dataset samples.json'))
for i in gen.index[:5]:
    print('Q:', rows[i]['prompt_bn'][:120])
    print('A (TigerLLM):', str(gen.loc[i, 'gen_answer'])[:160])
    print('R (given):   ', rows[i]['response_bn'][:160], '| label:', rows[i]['label'])
    print('---')

In [ ]:
# features for train + combined repeated CV — the go/no-go gate:
# closed_book Macro-F1 must clearly beat 0.345 (majority) for this to be worth submitting
%cd /content/repo
!cd src && python track_b_consistency.py features train
!cd src && python track_b_classifier.py

In [ ]:
# the slow part: 1,155 test closed-book rows (~1-2h on a T4).
# Fully resumable: answers flush to Drive every batch, so after ANY disconnect just
# re-run cells 1-3 and then this cell — it continues where it stopped.
%cd /content/repo
!cd src && python track_b_consistency.py gen test

In [ ]:
%cd /content/repo
!cd src && python track_b_consistency.py features test

In [ ]:
# build + validate the submission
%cd /content/repo
!cd src && python combine_and_predict.py nli_ctx_trackb_cb.csv

In [ ]:
# persist the submission to Drive (caches are already there via the symlink)
!mkdir -p "{DRIVE_DIR}/submissions"
!cp submissions/nli_ctx_trackb_cb.csv "{DRIVE_DIR}/submissions/"
!ls -la "{DRIVE_DIR}/cache" "{DRIVE_DIR}/submissions"

Done. Report the CV numbers (overall / context / closed_book ± std) back before submitting —
only submit if closed-book clearly beats the 0.345 majority baseline:

```
kaggle competitions submit -c bengali-hallucination -f submissions/nli_ctx_trackb_cb.csv \
  -m "Track A NLI + Track B TigerLLM consistency classifier"
```